# Teste isolado — ARTESP (Sala de Imprensa)

Fonte candidata: **ARTESP - Agência de Transporte do Estado de São Paulo**,
setor Transporte. Já existia placeholder em `controle_fontes`
(`source_id='—'`, `status='Não iniciada'`, `importancia_original='Média'`).
Notebook **descartável** (Fase 1) — sem dispatcher, sem
`atualizar_status_fonte`, sem gravar nada em produção.

URL fornecida: `https://www.artesp.sp.gov.br/artesp` (home institucional).
Página de notícias identificada por navegação: `Canais de Comunicação →
Sala de Imprensa`
(`/artesp/canais-de-comunicacao/sala-de-imprensa`).

In [ ]:
%pip install --quiet httpx
dbutils.library.restartPython()

In [ ]:
import time
import httpx

BASE = "https://www.artesp.sp.gov.br"
HOME = f"{BASE}/artesp"
SALA_IMPRENSA = f"{BASE}/artesp/canais-de-comunicacao/sala-de-imprensa"

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/120.0 Safari/537.36"
)
HEADERS = {"User-Agent": USER_AGENT, "Accept-Language": "pt-BR,pt;q=0.9"}

## Teste 1 — a mesma URL responde de forma inconsistente

Faz a mesma requisição várias vezes, com um `Client` novo (cookie jar
limpo) a cada tentativa, pra não presumir que é só falta de cookie de
sessão. Confirma antes de assumir: não presume que uma resposta 200 é
sempre conteúdo real — precisa checar o corpo.

In [ ]:
def eh_interstitial_imperva(corpo: str) -> bool:
    return "Pardon Our Interruption" in corpo or "reeseSkipExpirationCheck" in corpo


def eh_portlet_com_erro(corpo: str) -> bool:
    return "lrpError" in corpo and "Configura" in corpo


resultados = []
for tentativa in range(1, 6):
    with httpx.Client(headers=HEADERS, timeout=30, follow_redirects=True) as client:
        resp = client.get(SALA_IMPRENSA)
        corpo = resp.text
        if eh_interstitial_imperva(corpo):
            classificacao = "WAF (Imperva/Incapsula) -- challenge JS, sem conteúdo real"
        elif eh_portlet_com_erro(corpo):
            classificacao = "conteúdo real, MAS portlet de notícias quebrado ('Configuração inválida')"
        else:
            classificacao = "conteúdo real, sem erro aparente"
        resultados.append((tentativa, resp.status_code, len(corpo), classificacao))
        print(f"tentativa {tentativa}: HTTP {resp.status_code}, {len(corpo)} chars -> {classificacao}")
    time.sleep(4)

print("\nResumo:")
for t, status, tamanho, classificacao in resultados:
    print(f"  [{t}] {status} / {tamanho}c / {classificacao}")

## Teste 2 — `robots.txt` não é um robots.txt de verdade

`/robots.txt` devolve o mesmo HTML da home institucional (mesmo tamanho,
mesma estrutura de `<head>`), não um arquivo de diretivas -- a rota cai
no catch-all do WCM/portal em vez de servir um arquivo estático. Não dá
pra usar isso pra decidir permissão de scraping; documentando só como
achado de diagnóstico, igual foi feito para AGRESE.

In [ ]:
with httpx.Client(headers=HEADERS, timeout=30, follow_redirects=True) as client:
    resp_robots = client.get(f"{BASE}/robots.txt")
    resp_home = client.get(HOME)

print(f"/robots.txt  -> HTTP {resp_robots.status_code}, {len(resp_robots.text)} chars")
print(f"/artesp      -> HTTP {resp_home.status_code}, {len(resp_home.text)} chars")
print(f"\nrobots.txt parece HTML de página (não diretivas Disallow/Allow)? "
      f"{'User-agent' not in resp_robots.text and '<html' in resp_robots.text.lower()}")

## Conclusão da Fase 1

**Dois problemas independentes, empilhados:**

1. **WAF ativo (Imperva/Incapsula)**: a mesma URL, com o mesmo `User-Agent`
   de navegador, alterna entre devolver o HTML real da página e devolver
   a página de challenge JS ("Pardon Our Interruption", cookies
   `visid_incap_*`/`incap_ses_*`/`nlbi_*`, `reeseSkipExpirationCheck`) --
   sempre com HTTP 200, nunca 403/429, o que torna difícil até detectar o
   bloqueio só pelo status code. Um cliente HTTP simples (`httpx`, sem
   executar JS) não tem garantia de receber conteúdo real em nenhuma
   tentativa dada -- mesmo padrão de risco do bloqueio da ANAC (WAF
   F5/Shape), só que com vendor diferente (Imperva) e um comportamento
   mais intermitente (ANAC bloqueia praticamente sempre; aqui alterna).
2. **Quando o WAF deixa passar**: a própria página `Sala de Imprensa` do
   site tem um portlet de listagem quebrado -- exibe um aviso nativo do
   CMS ("Configuração inválida localizada. Entre em contato com o
   administrador.") em vez da lista de notícias/releases. Ou seja, mesmo
   contornando o WAF, não haveria lista de itens pra extrair no estado
   atual do site -- bug do lado da ARTESP, não do nosso lado.
3. `/robots.txt` não é um robots.txt real (devolve o HTML da home) --
   não dá pra usar como sinal de permissão/proibição.

**Avaliação para a Fase 2**: não implementar scraping por ora. Mesmo
raciocínio do bloqueio da ANAC -- WAFs desse tipo (challenge JS) não têm
garantia de serem contornáveis nem por navegador automatizado (Selenium),
e aqui ainda existe o problema adicional e independente do portlet
quebrado no próprio site. Vai direto para bloqueio formal na Fase 3.